In [1]:
import pandas as pd
import bisect

In [2]:
tickers = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL', 'MSFT', 'IBM', 'ORCL', 'NVDA', 'INTC']

In [3]:
def get_date(series, offset=8):
    series = pd.to_datetime(series, format='%Y-%m-%d %H:%M:%S')
    series = series + pd.Timedelta(hours=offset)
    return series.dt.date.astype(str)

In [4]:
stock_ds = {}
for key in tickers:
    stock_ds[key] = pd.read_csv(f'../dataset/stocks/{key}.csv')
    stock_ds[key] = stock_ds[key].sort_values(by='time')
    stock_ds[key]['ticker'] = key
    stock_ds[key]['date'] = get_date(stock_ds[key]['time'], 0)
    stock_ds[key]['ratio'] = stock_ds[key]['close'] / stock_ds[key]['close'].shift(1) - 1.0

stock_df = pd.concat(stock_ds.values(), axis=0, ignore_index=True)
stock_df = stock_df[~stock_df['ratio'].isna()].sort_values(by='time').reset_index(drop=True)

In [5]:
trade_days = sorted(stock_df['date'].unique())

In [6]:
def map_trading_day(trade_days):
    def inner(date_str: str) -> str | None:
        idx = bisect.bisect_left(trade_days, date_str)
        if idx < len(trade_days):
            return trade_days[idx]
        else:
            return None
    return inner

In [7]:
news_ds = {}
for key in tickers:
    news_ds[key] = pd.read_csv(f'../dataset/news/{key}_main.csv')
    news_ds[key] = news_ds[key].sort_values(by='publish_time')
    news_ds[key]['date'] = get_date(news_ds[key]['publish_time'])

news_df = pd.concat(news_ds.values(), axis=0, ignore_index=True)
news_df = news_df.sort_values(by='publish_time').drop_duplicates(subset=['id'])
news_df['date'] = news_df['date'].map(map_trading_day(trade_days))
news_df = news_df[~news_df['date'].isna()].reset_index(drop=True)

In [8]:
relat_all_df = pd.read_csv(f'../dataset/news/relation_all.csv')
relat_df = relat_all_df[relat_all_df['source_ticker']==relat_all_df['ticker']]
relat_df = relat_df.sort_values(by='time').drop_duplicates(subset=['news_id'])
relat_df = relat_df.reset_index(drop=True)
relat_df['upd_date'] = get_date(relat_df['time'])

In [9]:
relat_df = relat_df[['news_id', 'ticker', 'sentiment', 'sentiment_reasoning', 'upd_date']] \
    .merge(news_df[['id', 'description', 'date']], left_on='news_id', right_on='id', how='inner')
# (relat_df['date'] == relat_df['upd_date']).mean() == 1.0
relat_df = relat_df[['ticker', 'sentiment', 'sentiment_reasoning', 'description', 'date']]

In [10]:
mk_df = relat_df.merge(stock_df[['ticker', 'date', 'ratio']], on=['ticker', 'date'], how='inner')
mk_df = mk_df.dropna().reset_index(drop=True)
mk_df.to_csv(f'../dataset/market.csv', index=False)

In [11]:
mk_train = mk_df.sample(frac=0.9, random_state=42)
mk_test = mk_df.drop(mk_train.index)
mk_train.to_csv(f'../dataset/market_train.csv', index=False)
mk_test.to_csv(f'../dataset/market_test.csv', index=False)